In [1]:
# Loading packages
import pandas as pd
import numpy as np
import statsmodels.formula.api as sm
import matplotlib.pyplot as plt

In [2]:
# Import Data
data_url = "https://raw.githubusercontent.com/datamisc/ts-2020/main/data.csv"
anes_data  = pd.read_csv(data_url, compression='gzip')


/tmp/ipykernel_903441/4062180276.py:3: DtypeWarning: Columns (15,17,18,19,21,22,23,25,26,27,29,30,31,33,34,35,37,38,1508,1509) have mixed types. Specify dtype option on import or set low_memory=False.
  anes_data  = pd.read_csv(data_url, compression='gzip')


In [3]:
# Selecting relevant variables
my_vars = [
    "V201033",  # vote-int
    "V201507x",  # age
    "V201600",  # sex
    "V201511x",  # educ
    "V201617x",  # income
    "V201231x",  # party-id
    "V201200",  # idl
    "V201151",  # rate-biden
]

df = anes_data[my_vars]
df.columns = ['vote_int', 'age', 'sex', 'educ', 'income', 'party_id', 'ideology', 'approval_rating']

In [4]:
# Handling missing values & others
df = df[df >= 0]
df = df.dropna()
df = df[df['ideology'].between(1,7)]

In [13]:
# Preparing the explained variable (Y)
df['vote_biden'] = df['vote_int'] == 1
df['vote_biden'] = df['vote_biden'].astype(int)


In [18]:
# Preparing the model formula
# DV ~ IV + CV
formula = "vote_biden ~ age + sex + educ + income + party_id + ideology + approval_rating"


In [19]:
# Fit the logistic regression model
model = sm.logit(formula=formula, data=df).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.158626
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:             vote_biden   No. Observations:                 5582
Model:                          Logit   Df Residuals:                     5574
Method:                           MLE   Df Model:                            7
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                  0.7702
Time:                        12:55:46   Log-Likelihood:                -885.45
converged:                       True   LL-Null:                       -3853.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.5075      0.427     -1.190      0.234      -1.344       0.329
age           

Notice the sex, educ, and party_id coefficient values ? 

In [ ]:
# Fixing the formula
formula2 = "vote_biden ~ age + C(sex) + C(educ) + income + C(party_id) + ideology + approval_rating"

In [ ]:
model2 = sm.logit(formula=formula2, data=df).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.155659
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:             vote_biden   No. Observations:                 5582
Model:                          Logit   Df Residuals:                     5566
Method:                           MLE   Df Model:                           15
Date:                Mon, 02 Mar 2026   Pseudo R-squ.:                  0.7745
Time:                        12:56:22   Log-Likelihood:                -868.89
converged:                       True   LL-Null:                       -3853.6
Covariance Type:            nonrobust   LLR p-value:                     0.000
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.0095      0.523     -1.931      0.054      -2.034       0.015
C(sex

Holding all other variables constant, females have 0.2325 lower log-odds of voting for Biden than males (~20% lower odds than males). 

This result is unusual, as in most U.S. election data (e.g., American National Election Studies), women tend to be more Democratic-leaning than men. **BUT,** notice that the result is not significant!

In [ ]:
# Creating predicted probabilities based on 'age'
age_values = np.linspace(df['age'].min(), df['age'].max(), 100)
# Create a DataFrame to store predictions
pred_df = pd.DataFrame({'age': age_values})
# Repeat other variables as their mean or mode for prediction
for col in X.columns:
    if col not in ['age', 'const']:
        pred_df[col] = X[col].mean()
# Add constant
pred_df = sm.add_constant(pred_df, has_constant='add')

# Predicted probabilities
pred_probs = model.predict(pred_df)

In [ ]:
#ideology Plotting
plt.figure(figsize=(10, 10))
plt.plot(age_values, pred_probs, label='Predicted Probability', linewidth=3)
plt.title('Predicted Probability of Voting Intention by Age', fontsize=30)
plt.xlabel('Age', fontsize=25)
plt.ylabel('Predicted Probability', fontsize=25)
plt.grid(True)
plt.legend(fontsize=20)
plt.yticks(fontsize=20)
plt.xticks(fontsize=20)
plt.legend(fontsize=20)